In [5]:
import pandas as pd
import os

# Define file paths
csv_file_path = '../data/processed/clinical_hnsc_all.csv'
txt_file_path = '../data/raw/Subtype_Immune_Model_Based.txt'
output_dir = '../data/processed'
output_file_path = os.path.join(output_dir, 'clinical_hnsc_all_sb.csv')

# 1. Load the data
df_csv = pd.read_csv(csv_file_path)
df_txt = pd.read_csv(txt_file_path, sep='\t')

# 2. Create a matching key in the CSV DataFrame
# We take the first 15 characters to remove the trailing 'A', 'B', etc.
df_csv['sample_match'] = df_csv['sample'].str[:15]

# 3. Merge the DataFrames
# We use left_on and right_on since the matching columns have different names now
merged_df = pd.merge(
    df_csv, 
    df_txt[['sample', 'Subtype_Immune_Model_Based']], 
    left_on='sample_match', 
    right_on='sample', 
    how='left',
    suffixes=('', '_txt') # Prevents the original 'sample' column from being renamed to 'sample_x'
)

# 4. Clean up columns
# Drop the temporary matching column and the extra 'sample' column from the txt file
merged_df = merged_df.drop(columns=['sample_match', 'sample_txt'])

# Rename the new feature column
merged_df = merged_df.rename(columns={'Subtype_Immune_Model_Based': 'immune_subtypes'})

# 5. Save the result
os.makedirs(output_dir, exist_ok=True)
#merged_df.to_csv(output_file_path, index=False)

#print(f"File successfully saved to: {output_file_path}")

In [6]:
import pandas as pd
import os

# Define file paths
csv_file_path = os.path.join('../data/processed', 'clinical_hnsc_all_sb.csv')
tsv_file_path = '../data/raw/TCGASubtype.20170308.tsv'

# 1. Load the data
df_csv = merged_df # Assuming this is the DataFrame from the previous step
df_tsv = pd.read_csv(tsv_file_path, sep='\t')

# 2. Select only the necessary columns from the TSV
columns_to_add = [
    'sampleID', # Keep this for the merge
    'Subtype_CNA',
    'Subtype_DNAmeth',
    'Subtype_Integrative',
    'Subtype_Selected',
    'Subtype_mRNA',
    'Subtype_miRNA',
    'Subtype_other',
    'Subtype_protein'
]
df_tsv_subset = df_tsv[columns_to_add]

# 3. Create a matching key in the CSV DataFrame
# Extract the first 15 characters (e.g., TCGA-HZ-7918-01) to match the TSV format
df_csv['sample_match'] = df_csv['sample'].str[:15]

# 4. Merge the DataFrames
merged_df = pd.merge(
    df_csv, 
    df_tsv_subset, 
    left_on='sample_match', 
    right_on='sampleID', 
    how='left'
)
merged_df
# 5. Clean up columns
# Drop the temporary 'sample_match' and the redundant 'sampleID' from the TSV


,Unnamed: 0,sample,sample_type.samples,tissue_type.samples,primary_site,ajcc_pathologic_stage.diagnoses,alcohol_history.exposures,gender.demographic,race.demographic,vital_status.demographic,...,sample_match,sampleID,Subtype_CNA,Subtype_DNAmeth,Subtype_Integrative,Subtype_Selected,Subtype_mRNA,Subtype_miRNA,Subtype_other,Subtype_protein
0,0,TCGA-BB-4227-01A,Primary Tumor,Tumor,Hypopharynx,Stage IVA,Yes,male,white,Alive,...,TCGA-BB-4227-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,TCGA-CR-6478-01A,Primary Tumor,Tumor,Tonsil,Unknown,Yes,female,white,Dead,...,TCGA-CR-6478-01,TCGA-CR-6478-01,2,hyper-methylated,NaN,HNSC.Mesenchymal,Mesenchymal,3,Paradigma.4,NaN
2,2,TCGA-BB-4228-01A,Primary Tumor,Tumor,Base of tongue,Stage III,Yes,male,white,Alive,...,TCGA-BB-4228-01,TCGA-BB-4228-01,3,CpG island hyper-methylated,NaN,HNSC.Atypical,Atypical,3,Paradigma.5,NaN
3,3,TCGA-BA-4075-01A,Primary Tumor,Tumor,Other and unspecified parts of tongue,Stage III,Yes,male,black or african american,Dead,...,TCGA-BA-4075-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,TCGA-CX-7082-01A,Primary Tumor,Tumor,"Other and ill-defined sites in lip, oral cavit...",Stage IVA,Yes,male,white,Dead,...,TCGA-CX-7082-01,TCGA-CX-7082-01,1,normal-like,NaN,HNSC.Classical,Classical,1,Paradigma.2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,599,TCGA-CV-7242-11A,Solid Tissue Normal,Normal,Larynx,Stage II,Yes,female,white,Alive,...,TCGA-CV-7242-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
600,600,TCGA-TN-A7HL-01A,Primary Tumor,Tumor,Hypopharynx,Stage IVA,Yes,male,white,Alive,...,TCGA-TN-A7HL-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601,601,TCGA-CV-A6JZ-01A,Primary Tumor,Tumor,Other and unspecified parts of mouth,Stage II,Yes,male,white,Alive,...,TCGA-CV-A6JZ-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
602,602,TCGA-CV-5444-11A,Solid Tissue Normal,Normal,Larynx,Stage IVA,Yes,male,black or african american,Alive,...,TCGA-CV-5444-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
merged_df = merged_df.drop(columns=['sample_match', 'sampleID', 'Unnamed: 0'])

# 6. Save the result, overwriting the existing CSV file as requested
merged_df.to_csv(csv_file_path, index=False)

print(f"File successfully updated and saved to: {csv_file_path}")

File successfully updated and saved to: ../data/processed/clinical_hnsc_all_sb.csv


In [8]:
merged_df

,sample,sample_type.samples,tissue_type.samples,primary_site,ajcc_pathologic_stage.diagnoses,alcohol_history.exposures,gender.demographic,race.demographic,vital_status.demographic,tissue_source_site_id.tissue_source_site,...,primary_diagnosis.diagnoses,immune_subtypes,Subtype_CNA,Subtype_DNAmeth,Subtype_Integrative,Subtype_Selected,Subtype_mRNA,Subtype_miRNA,Subtype_other,Subtype_protein
0,TCGA-BB-4227-01A,Primary Tumor,Tumor,Hypopharynx,Stage IVA,Yes,male,white,Alive,a7da35ad-149e-5014-932b-37f188536bf8,...,"Squamous cell carcinoma, NOS",Wound Healing (Immune C1),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TCGA-CR-6478-01A,Primary Tumor,Tumor,Tonsil,Unknown,Yes,female,white,Dead,041ae4a7-3724-5587-82a9-8e09d3ee3cb8,...,"Squamous cell carcinoma, NOS",IFN-gamma Dominant (Immune C2),2,hyper-methylated,NaN,HNSC.Mesenchymal,Mesenchymal,3,Paradigma.4,NaN
2,TCGA-BB-4228-01A,Primary Tumor,Tumor,Base of tongue,Stage III,Yes,male,white,Alive,a7da35ad-149e-5014-932b-37f188536bf8,...,"Squamous cell carcinoma, NOS",Wound Healing (Immune C1),3,CpG island hyper-methylated,NaN,HNSC.Atypical,Atypical,3,Paradigma.5,NaN
3,TCGA-BA-4075-01A,Primary Tumor,Tumor,Other and unspecified parts of tongue,Stage III,Yes,male,black or african american,Dead,278e36e9-7faf-5043-ba90-2a4145ad0226,...,"Squamous cell carcinoma, NOS",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TCGA-CX-7082-01A,Primary Tumor,Tumor,"Other and ill-defined sites in lip, oral cavit...",Stage IVA,Yes,male,white,Dead,5184bb4b-3075-56fb-886f-7895a42e156b,...,"Squamous cell carcinoma, NOS",NaN,1,normal-like,NaN,HNSC.Classical,Classical,1,Paradigma.2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,TCGA-CV-7242-11A,Solid Tissue Normal,Normal,Larynx,Stage II,Yes,female,white,Alive,e0529816-35d4-5963-93c3-72c127e4e372,...,"Squamous cell carcinoma, NOS",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
600,TCGA-TN-A7HL-01A,Primary Tumor,Tumor,Hypopharynx,Stage IVA,Yes,male,white,Alive,0880c76d-8168-580c-82c0-aeb6ba104840,...,"Squamous cell carcinoma, NOS",IFN-gamma Dominant (Immune C2),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601,TCGA-CV-A6JZ-01A,Primary Tumor,Tumor,Other and unspecified parts of mouth,Stage II,Yes,male,white,Alive,e0529816-35d4-5963-93c3-72c127e4e372,...,"Squamous cell carcinoma, NOS",IFN-gamma Dominant (Immune C2),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
602,TCGA-CV-5444-11A,Solid Tissue Normal,Normal,Larynx,Stage IVA,Yes,male,black or african american,Alive,e0529816-35d4-5963-93c3-72c127e4e372,...,"Squamous cell carcinoma, NOS",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
